# Databricks SQL pipeline: DOHMH + PLUTO + NYPD

Ten notebook jest wersją pipeline dostosowaną do pracy w **Databricks Free Edition / serverless compute** z wykorzystaniem tabel w Unity Catalog oraz transformacji zapisanych jako zapytania SQL.

Założenia:
- pliki CSV są wgrane do volume: `/Volumes/workspace/default/project_files`,
- surowe CSV zostają przeniesione do bazy jako tabele Delta w `workspace.default`,
- etapy `TEMP1`, `NYPD_ZIP`, `TEMP2`, `TEMP3` tworzą tabele SQL,
- nie używamy `sparkContext`, `df.rdd`, `_jvm`, `_jsc`, `cache()` ani `persist()`,
- logujemy czas, status i liczbę rekordów dla każdego etapu.

Uwaga: etap `NYPD_ZIP` wykonuje geometrię point-in-polygon przez Python UDF zarejestrowany jako funkcja SQL `find_zip_sql`. Sama transformacja jest uruchamiana jako `CREATE OR REPLACE TABLE ... AS SELECT ...`.

In [ ]:
from time import perf_counter
from datetime import datetime
import json
import math
import re

from pyspark.sql import functions as F
from pyspark.sql import types as T

print("Spark version:", spark.version)

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
CURRENT_YEAR = datetime.now().year

print("Timestamp uruchomienia:", RUN_TS)

## 1. Konfiguracja katalogu, schematu, volume i trybu danych

W Databricks odpowiednikiem „bazy danych” jest **schema** w ramach **catalog**. W tym projekcie używamy:

```text
Catalog: workspace
Schema: default
Volume: project_files
```

Jeżeli uruchamiacie pipeline na małych danych testowych, zostawcie `USE_TEST_DATA = True`.  
Dla pełnych danych ustawcie `USE_TEST_DATA = False`.

In [ ]:
CATALOG = "workspace"
SCHEMA = "default"
VOLUME = "project_files"

BASE_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

USE_TEST_DATA = True

if USE_TEST_DATA:
    SOURCE_DIR = f"{BASE_VOLUME}/test"
    TABLE_PREFIX = "test_"
    RESULTS_DIR = f"{BASE_VOLUME}/results_sql_test"
else:
    SOURCE_DIR = BASE_VOLUME
    TABLE_PREFIX = ""
    RESULTS_DIR = f"{BASE_VOLUME}/results_sql"

DOHMH_CSV = f"{SOURCE_DIR}/DOHMH_latest.csv" if not USE_TEST_DATA else f"{SOURCE_DIR}/DOHMH_latest_test.csv"
PLUTO_CSV = f"{SOURCE_DIR}/PLUTO.csv" if not USE_TEST_DATA else f"{SOURCE_DIR}/PLUTO_test.csv"
NYPD_CSV = f"{SOURCE_DIR}/NYPD.csv" if not USE_TEST_DATA else f"{SOURCE_DIR}/NYPD_test.csv"
MODZCTA_CSV = f"{SOURCE_DIR}/MODZCTA.csv" if not USE_TEST_DATA else f"{SOURCE_DIR}/MODZCTA_test.csv"

DATABASE = f"{CATALOG}.{SCHEMA}"

def tbl(name: str) -> str:
    return f"{DATABASE}.{TABLE_PREFIX}{name}"

DOHMH_TABLE = tbl("raw_dohmh_sql")
PLUTO_TABLE = tbl("raw_pluto_sql")
NYPD_TABLE = tbl("raw_nypd_sql")
MODZCTA_TABLE = tbl("raw_modzcta_sql")

TEMP1_TABLE = tbl("temp1_latest_sql")
NYPD_ZIP_TABLE = tbl("nypd_zip_latest_sql")
TEMP2_TABLE = tbl("temp2_latest_sql")
TEMP3_TABLE = tbl("temp3_latest_sql")

print("BASE_VOLUME:", BASE_VOLUME)
print("SOURCE_DIR:", SOURCE_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("DATABASE:", DATABASE)
print("DOHMH_CSV:", DOHMH_CSV)
print("PLUTO_CSV:", PLUTO_CSV)
print("NYPD_CSV:", NYPD_CSV)
print("MODZCTA_CSV:", MODZCTA_CSV)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DATABASE}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

## 2. Przeniesienie CSV do bazy danych jako tabele Delta

W tej komórce każdy plik CSV z volume jest wczytywany, nazwy kolumn są normalizowane do formatu bez spacji i znaków specjalnych, a następnie dane są zapisywane jako zarządzane tabele Delta w `workspace.default`.

Po wykonaniu komórki w **Catalog Explorer** powinny być widoczne tabele:

- `raw_dohmh_sql`
- `raw_pluto_sql`
- `raw_nypd_sql`
- `raw_modzcta_sql`

Dla trybu testowego nazwy mają prefiks `test_`.

In [ ]:
def canonical_name(name: str) -> str:
    if name is None:
        return ""
    x = name.strip().replace("\ufeff", "")
    x = re.sub(r"[^A-Za-z0-9]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x.upper()


def load_csv_to_delta_table(path: str, table_name: str):
    print(f"\nŁadowanie CSV do tabeli: {table_name}")
    print("Źródło:", path)

    start = perf_counter()

    df = (
        spark.read
        .option("header", "true")
        .option("escape", "\"")
        .option("quote", "\"")
        .option("multiLine", "false")
        .csv(path)
    )

    for c in df.columns:
        df = df.withColumnRenamed(c, canonical_name(c))

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    rows = spark.sql(f"SELECT COUNT(*) AS rows FROM {table_name}").first()["rows"]
    elapsed = perf_counter() - start

    print("STATUS: OK")
    print(f"Tabela: {table_name}")
    print(f"Liczba rekordów: {rows}")
    print(f"Czas ładowania: {elapsed:.3f} s")

    return rows, elapsed


LOAD_RAW_TABLES = True

if LOAD_RAW_TABLES:
    raw_load_results = []
    raw_load_results.append(("DOHMH",) + load_csv_to_delta_table(DOHMH_CSV, DOHMH_TABLE))
    raw_load_results.append(("PLUTO",) + load_csv_to_delta_table(PLUTO_CSV, PLUTO_TABLE))
    raw_load_results.append(("NYPD",) + load_csv_to_delta_table(NYPD_CSV, NYPD_TABLE))
    raw_load_results.append(("MODZCTA",) + load_csv_to_delta_table(MODZCTA_CSV, MODZCTA_TABLE))

    print("\n===== PODSUMOWANIE ŁADOWANIA CSV DO BAZY =====")
    for name, rows, elapsed in raw_load_results:
        print(f"{name}: rows={rows}, czas={elapsed:.3f} s")
else:
    print("LOAD_RAW_TABLES = False. Pomijam ponowne ładowanie CSV do tabel.")

### Alternatywa SQL-only dla samego importu

Jeżeli prowadzący wymaga pokazania samego importu jako SQL, można w osobnej komórce SQL użyć funkcji `read_files`. Przykład:

```sql
CREATE OR REPLACE TABLE workspace.default.raw_dohmh_direct_sql AS
SELECT *
FROM read_files(
  '/Volumes/workspace/default/project_files/DOHMH_latest.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);
```

W naszym notebooku używamy funkcji Python do ładowania wyłącznie po to, żeby automatycznie oczyścić nazwy kolumn. Same etapy przetwarzania danych poniżej są realizowane przez SQL.

## 3. Funkcje pomocnicze do generowania SQL

Poniższe funkcje nie wykonują transformacji DataFrame. Służą tylko do wygenerowania poprawnych wyrażeń SQL zależnie od tego, które kolumny istnieją w tabelach wejściowych.

In [ ]:
def table_cols(table_name: str):
    return {field.name.upper(): field.name for field in spark.table(table_name).schema.fields}


def qcol(cols: dict, alias: str, *names: str, default: str = "''") -> str:
    for name in names:
        cn = canonical_name(name)
        if cn in cols:
            return f"{alias}.`{cols[cn]}`"
    return default


def only_digits_sql(expr: str) -> str:
    return f"regexp_replace(coalesce(CAST({expr} AS STRING), ''), '[^0-9]', '')"


def first_five_digits_sql(expr: str) -> str:
    d = only_digits_sql(expr)
    return f"CASE WHEN length({d}) >= 5 THEN substring({d}, 1, 5) ELSE {d} END"


def safe_double_sql(expr: str) -> str:
    txt = f"trim(coalesce(CAST({expr} AS STRING), ''))"
    return f"CASE WHEN {txt} RLIKE '^-?[0-9]+(\\\\.[0-9]+)?$' THEN CAST({txt} AS DOUBLE) ELSE NULL END"


def safe_int_sql(expr: str) -> str:
    txt = f"trim(coalesce(CAST({expr} AS STRING), ''))"
    return f"CASE WHEN {txt} RLIKE '^[0-9]+$' THEN CAST({txt} AS INT) ELSE NULL END"


def norm_bbl_sql(expr: str) -> str:
    d = only_digits_sql(expr)
    return f"""
    CASE
        WHEN length({d}) != 10 THEN ''
        WHEN {d} = '0000000000' THEN ''
        WHEN substring({d}, 2, 9) = '000000000' THEN ''
        ELSE {d}
    END
    """


def zscore_sql(value_expr: str, mean_expr: str, stdev_expr: str) -> str:
    return f"""
    CASE
        WHEN {stdev_expr} IS NULL OR {stdev_expr} = 0
          OR {value_expr} IS NULL OR {mean_expr} IS NULL
        THEN 0.0
        ELSE ({value_expr} - {mean_expr}) / {stdev_expr}
    END
    """


def format_duration(seconds: float) -> str:
    total_ms = int(round(seconds * 1000))
    ms = total_ms % 1000
    total_s = total_ms // 1000
    s = total_s % 60
    m = (total_s // 60) % 60
    h = total_s // 3600
    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"


def run_timed_sql(stage_name: str, sql_text: str, result_table: str):
    print(f"\n===== START ETAPU SQL: {stage_name} =====")
    print("Tabela wynikowa:", result_table)

    start = perf_counter()

    try:
        spark.sql(sql_text)
        rows = spark.sql(f"SELECT COUNT(*) AS rows FROM {result_table}").first()["rows"]
        elapsed = perf_counter() - start

        print("STATUS: OK")
        print(f"{stage_name} rows: {rows}")
        print(f"Czas trwania etapu {stage_name}: {format_duration(elapsed)} ({elapsed:.3f} s)")
        print(f"===== KONIEC ETAPU SQL: {stage_name} =====\n")

        return elapsed, rows, "OK"

    except Exception as exc:
        elapsed = perf_counter() - start

        print("STATUS: ERROR")
        print(f"Czas do błędu etapu {stage_name}: {format_duration(elapsed)} ({elapsed:.3f} s)")
        print("Typ błędu:", type(exc).__name__)
        print("Treść błędu:")
        print(str(exc)[:4000])
        print(f"===== KONIEC ETAPU SQL Z BŁĘDEM: {stage_name} =====\n")

        raise

## 4. Rejestracja funkcji SQL

Do SQL rejestrujemy:
- `normalize_address_sql(address)` — normalizacja adresów,
- `find_zip_sql(lat, lon)` — przypisanie punktu NYPD do ZIP/MODZCTA.

Druga funkcja jest potrzebna, ponieważ zwykły Spark SQL w Databricks Free Edition nie ma gotowej funkcji point-in-polygon dla geometrii MODZCTA.

In [ ]:
ADDRESS_ABBREVIATIONS = {
    "SAINT": "ST", "STREET": "ST", "STR": "ST",
    "AVENUE": "AVE", "AV": "AVE",
    "ROAD": "RD", "BOULEVARD": "BLVD", "DRIVE": "DR",
    "LANE": "LN", "COURT": "CT", "PLACE": "PL",
    "TERRACE": "TER", "PARKWAY": "PKWY", "HIGHWAY": "HWY",
    "EXPRESSWAY": "EXPY", "SQUARE": "SQ", "CIRCLE": "CIR",
    "NORTH": "N", "SOUTH": "S", "EAST": "E", "WEST": "W",
}


def normalize_address_py(address):
    if address is None:
        return ""
    x = str(address).upper()
    x = x.replace(".", " ").replace(",", " ").replace("#", " ").replace("/", " ")
    x = re.sub(r"[^A-Z0-9\-\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    if not x:
        return ""
    out = []
    for token in x.split(" "):
        token = re.sub(r"([0-9]+)(ST|ND|RD|TH)$", r"\1", token)
        out.append(ADDRESS_ABBREVIATIONS.get(token, token))
    return " ".join(out)


spark.udf.register("normalize_address_sql", normalize_address_py, T.StringType())

print("Zarejestrowano funkcję SQL: normalize_address_sql")

In [ ]:
def parse_wkt_geometry(wkt: str):
    if not wkt:
        return []
    text = str(wkt).strip()
    upper = text.upper()
    if not (upper.startswith("POLYGON") or upper.startswith("MULTIPOLYGON")):
        return []
    nums = re.findall(r"-?\d+(?:\.\d+)?\s+-?\d+(?:\.\d+)?", text)
    points = []
    for pair in nums:
        lon, lat = [float(x) for x in pair.split()[:2]]
        points.append((lat, lon))
    return [points] if len(points) >= 3 else []


def parse_geojson_geometry(text: str):
    if not text:
        return []
    try:
        obj = json.loads(text)
    except Exception:
        return []
    geom_type = str(obj.get("type", "")).upper()
    coords = obj.get("coordinates")
    if not coords:
        return []
    polygons = []
    if geom_type == "POLYGON":
        outer = coords[0] if coords else []
        polygons.append([(float(lat), float(lon)) for lon, lat in outer])
    elif geom_type == "MULTIPOLYGON":
        for poly in coords:
            outer = poly[0] if poly else []
            polygons.append([(float(lat), float(lon)) for lon, lat in outer])
    return [p for p in polygons if len(p) >= 3]


def parse_any_geometry(text: str):
    if not text:
        return []
    u = str(text).strip().upper()
    if u.startswith("POLYGON") or u.startswith("MULTIPOLYGON"):
        return parse_wkt_geometry(text)
    if "COORDINATES" in u:
        return parse_geojson_geometry(text)
    return []


def point_in_ring(lat: float, lon: float, ring):
    inside = False
    j = len(ring) - 1
    for i in range(len(ring)):
        lati, loni = ring[i]
        latj, lonj = ring[j]
        crosses = ((loni > lon) != (lonj > lon)) and (
            lat < (latj - lati) * (lon - loni) / ((lonj - loni) or 1e-12) + lati
        )
        if crosses:
            inside = not inside
        j = i
    return inside


def ring_bbox(ring):
    lats = [p[0] for p in ring]
    lons = [p[1] for p in ring]
    return (min(lats), max(lats), min(lons), max(lons))


def build_spatial_grid(areas, grid_size=0.01):
    grid = {}
    polygon_count = 0

    for zip_code, polygons in areas:
        for ring in polygons:
            if len(ring) < 3:
                continue

            bbox = ring_bbox(ring)
            min_lat, max_lat, min_lon, max_lon = bbox

            min_i = math.floor(min_lat / grid_size)
            max_i = math.floor(max_lat / grid_size)
            min_j = math.floor(min_lon / grid_size)
            max_j = math.floor(max_lon / grid_size)

            payload = (zip_code, bbox, ring)

            for i in range(min_i, max_i + 1):
                for j in range(min_j, max_j + 1):
                    grid.setdefault((i, j), []).append(payload)

            polygon_count += 1

    return grid, polygon_count


def grid_key(lat, lon, grid_size=0.01):
    return (math.floor(lat / grid_size), math.floor(lon / grid_size))


def register_find_zip_sql_udf():
    mod_cols = table_cols(MODZCTA_TABLE)

    zip_candidates = ["MODZCTA", "ZCTA", "ZIPCODE", "ZIP_CODE", "POSTCODE", "ZIP"]
    geom_candidates = ["THE_GEOM", "GEOMETRY", "GEOM", "SHAPE", "WKT", "NEW_GEOREFERENCED_COLUMN", "GEOCODED_COLUMN"]

    zip_col = next((canonical_name(c) for c in zip_candidates if canonical_name(c) in mod_cols), None)
    geom_col = next((canonical_name(c) for c in geom_candidates if canonical_name(c) in mod_cols), None)

    if zip_col is None or geom_col is None:
        raise ValueError(f"Nie znaleziono kolumn ZIP/geometrii w {MODZCTA_TABLE}. Kolumny: {list(mod_cols.values())}")

    areas = []

    rows = spark.sql(f"""
        SELECT `{mod_cols[zip_col]}` AS ZIP_RAW, `{mod_cols[geom_col]}` AS GEOM_RAW
        FROM {MODZCTA_TABLE}
    """).collect()

    for row in rows:
        zip_code = re.sub(r"[^0-9]", "", str(row["ZIP_RAW"] or ""))[:5]
        polygons = parse_any_geometry(row["GEOM_RAW"])
        polygons = [p for p in polygons if len(p) >= 3]

        if zip_code and polygons:
            areas.append((zip_code, polygons))

    if not areas:
        raise ValueError("Nie wczytano żadnych poligonów MODZCTA.")

    grid_size = 0.01
    spatial_grid, polygon_count = build_spatial_grid(areas, grid_size=grid_size)

    print(f"MODZCTA: obszary ZIP={len(areas)}, poligony={polygon_count}, komórki siatki={len(spatial_grid)}")

    def find_zip_py(lat, lon):
        if lat is None or lon is None:
            return ""

        try:
            lat_value = float(lat)
            lon_value = float(lon)
        except Exception:
            return ""

        candidates = spatial_grid.get(grid_key(lat_value, lon_value, grid_size), [])

        for zip_code, bbox, ring in candidates:
            min_lat, max_lat, min_lon, max_lon = bbox

            if lat_value < min_lat or lat_value > max_lat or lon_value < min_lon or lon_value > max_lon:
                continue

            if point_in_ring(lat_value, lon_value, ring):
                return zip_code

        return ""

    spark.udf.register("find_zip_sql", find_zip_py, T.StringType())

    print("Zarejestrowano funkcję SQL: find_zip_sql")


register_find_zip_sql_udf()

## 5. Etap SQL TEMP1

Etap `TEMP1`:
- czyści dane DOHMH,
- usuwa rekordy z fikcyjną datą inspekcji `1900-01-01`,
- wyznacza agregaty po ZIP i BORO,
- łączy dane z PLUTO po BBL albo po znormalizowanym adresie,
- zapisuje wynik jako tabelę SQL.

In [ ]:
def build_temp1_sql():
    dcols = table_cols(DOHMH_TABLE)
    pcols = table_cols(PLUTO_TABLE)

    inspection_expr = f"upper(trim(coalesce(CAST({qcol(dcols, 'd', 'INSPECTION_DATE', 'INSPECTION DATE')} AS STRING), '')))"
    score_expr = safe_double_sql(qcol(dcols, "d", "SCORE"))
    zip_expr = first_five_digits_sql(qcol(dcols, "d", "ZIPCODE", "ZIP CODE"))
    camis_expr = only_digits_sql(qcol(dcols, "d", "CAMIS"))
    building_expr = f"trim(coalesce(CAST({qcol(dcols, 'd', 'BUILDING')} AS STRING), ''))"
    street_expr = f"trim(coalesce(CAST({qcol(dcols, 'd', 'STREET')} AS STRING), ''))"
    boro_expr = f"upper(trim(coalesce(CAST({qcol(dcols, 'd', 'BORO', 'BOROUGH')} AS STRING), '')))"
    cuisine_expr = f"upper(trim(coalesce(CAST({qcol(dcols, 'd', 'CUISINE_DESCRIPTION', 'CUISINE DESCRIPTION')} AS STRING), '')))"
    bbl_expr = norm_bbl_sql(qcol(dcols, "d", "BBL"))

    pluto_address_expr = f"normalize_address_sql({qcol(pcols, 'p', 'ADDRESS')})"
    pluto_zip_expr = first_five_digits_sql(qcol(pcols, "p", "POSTCODE", "ZIPCODE", "ZIP CODE"))
    pluto_bbl_expr = norm_bbl_sql(qcol(pcols, "p", "BBL"))
    pluto_landuse_expr = f"trim(coalesce(CAST({qcol(pcols, 'p', 'LANDUSE', 'LAND USE')} AS STRING), ''))"
    pluto_year_expr = safe_int_sql(only_digits_sql(qcol(pcols, "p", "YEARBUILT", "YEAR BUILT")))

    return f"""
CREATE OR REPLACE TABLE {TEMP1_TABLE} AS
WITH dohmh_base AS (
    SELECT
        {inspection_expr} AS INSPECTION_DATE_NORM,
        {zip_expr} AS ZIPCODE,
        {camis_expr} AS CAMIS,
        {score_expr} AS SCORE,
        CASE WHEN {cuisine_expr} = '' THEN 'UNKNOWN' ELSE {cuisine_expr} END AS CUISINE_DESCRIPTION,
        {building_expr} AS BUILDING,
        {street_expr} AS STREET,
        {boro_expr} AS BORO,
        {bbl_expr} AS BBL,
        normalize_address_sql(concat_ws(' ', {building_expr}, {street_expr})) AS ADDRESS
    FROM {DOHMH_TABLE} d
),
dohmh AS (
    SELECT ZIPCODE, CAMIS, SCORE, CUISINE_DESCRIPTION, ADDRESS, BORO, BBL
    FROM dohmh_base
    WHERE NOT (
        INSPECTION_DATE_NORM = '01/01/1900'
        OR INSPECTION_DATE_NORM = '1900-01-01T00:00:00.000'
        OR startswith(INSPECTION_DATE_NORM, '1900-01-01')
    )
      AND ZIPCODE <> ''
      AND CAMIS <> ''
      AND SCORE IS NOT NULL
      AND BUILDING <> ''
      AND STREET <> ''
      AND BORO <> ''
),
zip_stats AS (
    SELECT
        ZIPCODE,
        COUNT(DISTINCT CAMIS) AS NUMBER_PER_ZIP,
        AVG(SCORE) AS AVG_SCORE_ZIP
    FROM dohmh
    GROUP BY ZIPCODE
),
boro_stats AS (
    SELECT
        BORO,
        COUNT(DISTINCT CAMIS) AS NUMBER_PER_BORO
    FROM dohmh
    GROUP BY BORO
),
boro_cd_stats AS (
    SELECT
        BORO,
        CUISINE_DESCRIPTION,
        AVG(SCORE) AS AVG_SCORE_BORO_CD
    FROM dohmh
    GROUP BY BORO, CUISINE_DESCRIPTION
),
pluto_base AS (
    SELECT
        {pluto_address_expr} AS ADDRESS_NORM,
        {pluto_zip_expr} AS ZIPCODE,
        {pluto_bbl_expr} AS BBL,
        {pluto_landuse_expr} AS LANDUSE_RAW,
        {pluto_year_expr} AS YEARBUILT_RAW
    FROM {PLUTO_TABLE} p
),
pluto_bbl AS (
    SELECT
        concat('B|', BBL) AS JOIN_KEY,
        max(CASE WHEN LANDUSE_RAW <> '' THEN LANDUSE_RAW ELSE NULL END) AS LANDUSE,
        max(YEARBUILT_RAW) AS YEARBUILT
    FROM pluto_base
    WHERE BBL <> ''
    GROUP BY concat('B|', BBL)
),
pluto_addr AS (
    SELECT
        concat_ws('|', 'A', ZIPCODE, ADDRESS_NORM) AS JOIN_KEY,
        max(CASE WHEN LANDUSE_RAW <> '' THEN LANDUSE_RAW ELSE NULL END) AS LANDUSE,
        max(YEARBUILT_RAW) AS YEARBUILT
    FROM pluto_base
    WHERE ZIPCODE <> '' AND ADDRESS_NORM <> ''
    GROUP BY concat_ws('|', 'A', ZIPCODE, ADDRESS_NORM)
),
pluto_join AS (
    SELECT
        JOIN_KEY,
        max(LANDUSE) AS LANDUSE,
        max(YEARBUILT) AS YEARBUILT
    FROM (
        SELECT * FROM pluto_bbl
        UNION ALL
        SELECT * FROM pluto_addr
    )
    GROUP BY JOIN_KEY
),
dohmh_join AS (
    SELECT
        *,
        CASE
            WHEN BBL <> '' THEN concat('B|', BBL)
            ELSE concat_ws('|', 'A', ZIPCODE, ADDRESS)
        END AS JOIN_KEY
    FROM dohmh
)
SELECT
    dj.ZIPCODE,
    dj.CAMIS,
    CAST(dj.SCORE AS DOUBLE) AS SCORE,
    dj.CUISINE_DESCRIPTION,
    dj.ADDRESS,
    dj.BORO,
    CAST(zs.NUMBER_PER_ZIP AS BIGINT) AS NUMBER_PER_ZIP,
    CAST(bs.NUMBER_PER_BORO AS BIGINT) AS NUMBER_PER_BORO,
    round(zs.AVG_SCORE_ZIP, 4) AS AVG_SCORE_ZIP,
    round(bcs.AVG_SCORE_BORO_CD, 4) AS AVG_SCORE_BORO_CD,
    coalesce(pj.LANDUSE, '') AS LANDUSE,
    CAST(pj.YEARBUILT AS INT) AS YEARBUILT
FROM dohmh_join dj
LEFT JOIN zip_stats zs ON dj.ZIPCODE = zs.ZIPCODE
LEFT JOIN boro_stats bs ON dj.BORO = bs.BORO
LEFT JOIN boro_cd_stats bcs
    ON dj.BORO = bcs.BORO
   AND dj.CUISINE_DESCRIPTION = bcs.CUISINE_DESCRIPTION
LEFT JOIN pluto_join pj ON dj.JOIN_KEY = pj.JOIN_KEY
"""


temp1_sql = build_temp1_sql()

## 6. Etap SQL NYPD_ZIP

Etap `NYPD_ZIP`:
- przygotowuje dane NYPD,
- filtruje rekordy bez współrzędnych albo poza analizowanym zakresem,
- wywołuje funkcję SQL `find_zip_sql(LATITUDE, LONGITUDE)`,
- zapisuje wynik jako tabelę SQL.

In [ ]:
def build_nypd_zip_sql():
    ncols = table_cols(NYPD_TABLE)

    cmplnt_expr = f"trim(coalesce(CAST({qcol(ncols, 'n', 'CMPLNT_NUM')} AS STRING), ''))"
    ofns_expr = f"upper(trim(coalesce(CAST({qcol(ncols, 'n', 'OFNS_DESC')} AS STRING), '')))"
    boro_expr = f"upper(trim(coalesce(CAST({qcol(ncols, 'n', 'BORO_NM')} AS STRING), '')))"
    lat_expr = safe_double_sql(qcol(ncols, "n", "LATITUDE"))
    lon_expr = safe_double_sql(qcol(ncols, "n", "LONGITUDE"))

    return f"""
CREATE OR REPLACE TABLE {NYPD_ZIP_TABLE} AS
WITH nypd_prepared AS (
    SELECT
        {cmplnt_expr} AS CMPLNT_NUM,
        {ofns_expr} AS OFNS_DESC,
        {boro_expr} AS BORO_NM,
        {lat_expr} AS LATITUDE,
        {lon_expr} AS LONGITUDE
    FROM {NYPD_TABLE} n
),
filtered AS (
    SELECT *
    FROM nypd_prepared
    WHERE CMPLNT_NUM <> ''
      AND OFNS_DESC <> ''
      AND LATITUDE IS NOT NULL
      AND LONGITUDE IS NOT NULL
      AND LATITUDE >= 40.0
      AND LATITUDE <= 41.2
      AND LONGITUDE >= -75.2
      AND LONGITUDE <= -72.8
),
matched AS (
    SELECT
        CMPLNT_NUM,
        find_zip_sql(LATITUDE, LONGITUDE) AS ZIPCODE,
        OFNS_DESC,
        BORO_NM,
        round(LATITUDE, 6) AS LATITUDE,
        round(LONGITUDE, 6) AS LONGITUDE
    FROM filtered
)
SELECT
    CMPLNT_NUM,
    ZIPCODE,
    OFNS_DESC,
    BORO_NM,
    LATITUDE,
    LONGITUDE
FROM matched
WHERE ZIPCODE <> ''
"""


nypd_zip_sql = build_nypd_zip_sql()

## 7. Etap SQL TEMP2

Etap `TEMP2`:
- łączy wynik `TEMP1` z agregatami przestępczości,
- wyznacza dominantę typu przestępstwa po ZIP,
- oblicza średnie i odchylenia standardowe,
- tworzy wskaźniki pośrednie.

In [ ]:
def build_temp2_sql():
    return f"""
CREATE OR REPLACE TABLE {TEMP2_TABLE} AS
WITH temp1 AS (
    SELECT
        {first_five_digits_sql('ZIPCODE')} AS ZIPCODE,
        CAMIS,
        {safe_double_sql('SCORE')} AS SCORE,
        upper(trim(coalesce(CAST(CUISINE_DESCRIPTION AS STRING), ''))) AS CUISINE_DESCRIPTION,
        ADDRESS,
        upper(trim(coalesce(CAST(BORO AS STRING), ''))) AS BORO,
        {safe_double_sql('NUMBER_PER_ZIP')} AS NUMBER_PER_ZIP,
        {safe_double_sql('NUMBER_PER_BORO')} AS NUMBER_PER_BORO,
        {safe_double_sql('AVG_SCORE_ZIP')} AS AVG_SCORE_ZIP,
        {safe_double_sql('AVG_SCORE_BORO_CD')} AS AVG_SCORE_BORO_CD,
        LANDUSE,
        {safe_int_sql('YEARBUILT')} AS YEARBUILT
    FROM {TEMP1_TABLE}
),
zip_unique AS (
    SELECT DISTINCT ZIPCODE, NUMBER_PER_ZIP, AVG_SCORE_ZIP
    FROM temp1
),
boro_unique AS (
    SELECT DISTINCT BORO, NUMBER_PER_BORO
    FROM temp1
),
stdev_per_zip AS (
    SELECT coalesce(stddev_pop(NUMBER_PER_ZIP), 0.0) AS STDEV_PER_ZIP
    FROM zip_unique
),
stdev_per_boro AS (
    SELECT coalesce(stddev_pop(NUMBER_PER_BORO), 0.0) AS STDEV_PER_BORO
    FROM boro_unique
),
crime_base AS (
    SELECT DISTINCT
        {first_five_digits_sql('ZIPCODE')} AS ZIPCODE,
        trim(coalesce(CAST(CMPLNT_NUM AS STRING), '')) AS CMPLNT_NUM,
        upper(trim(coalesce(CAST(OFNS_DESC AS STRING), ''))) AS OFNS_DESC
    FROM {NYPD_ZIP_TABLE}
    WHERE {first_five_digits_sql('ZIPCODE')} <> ''
      AND trim(coalesce(CAST(CMPLNT_NUM AS STRING), '')) <> ''
      AND upper(trim(coalesce(CAST(OFNS_DESC AS STRING), ''))) <> ''
),
crime_count AS (
    SELECT
        ZIPCODE,
        COUNT(DISTINCT CMPLNT_NUM) AS COUNT_CRIME_PER_ZIP
    FROM crime_base
    GROUP BY ZIPCODE
),
offense_counts AS (
    SELECT
        ZIPCODE,
        OFNS_DESC,
        COUNT(*) AS OFFENSE_COUNT
    FROM crime_base
    GROUP BY ZIPCODE, OFNS_DESC
),
dominant_ranked AS (
    SELECT
        ZIPCODE,
        OFNS_DESC,
        row_number() OVER (
            PARTITION BY ZIPCODE
            ORDER BY OFFENSE_COUNT DESC, OFNS_DESC ASC
        ) AS rn
    FROM offense_counts
),
dominant AS (
    SELECT ZIPCODE, OFNS_DESC AS DOMINANT_CRIME_TYPE
    FROM dominant_ranked
    WHERE rn = 1
),
crime_aligned AS (
    SELECT
        z.ZIPCODE,
        coalesce(cc.COUNT_CRIME_PER_ZIP, 0) AS COUNT_CRIME_PER_ZIP,
        coalesce(d.DOMINANT_CRIME_TYPE, '') AS DOMINANT_CRIME_TYPE
    FROM (SELECT DISTINCT ZIPCODE FROM temp1) z
    LEFT JOIN crime_count cc ON z.ZIPCODE = cc.ZIPCODE
    LEFT JOIN dominant d ON z.ZIPCODE = d.ZIPCODE
),
crime_stats AS (
    SELECT
        coalesce(avg(COUNT_CRIME_PER_ZIP), 0.0) AS AVG_CRIME_PER_ZIP,
        coalesce(stddev_pop(COUNT_CRIME_PER_ZIP), 0.0) AS STDEV_CRIME_PER_ZIP
    FROM crime_aligned
),
score_stats AS (
    SELECT
        coalesce(avg(AVG_SCORE_ZIP), 0.0) AS MEAN_AVG_SCORE_ZIP,
        coalesce(stddev_pop(AVG_SCORE_ZIP), 0.0) AS STDEV_AVG_SCORE_ZIP
    FROM zip_unique
)
SELECT
    t.ZIPCODE,
    t.CAMIS,
    t.SCORE,
    t.CUISINE_DESCRIPTION,
    t.ADDRESS,
    t.BORO,
    CAST(t.NUMBER_PER_ZIP AS BIGINT) AS NUMBER_PER_ZIP,
    round(sz.STDEV_PER_ZIP, 4) AS STDEV_PER_ZIP,
    CAST(t.NUMBER_PER_BORO AS BIGINT) AS NUMBER_PER_BORO,
    round(sb.STDEV_PER_BORO, 4) AS STDEV_PER_BORO,
    round(t.AVG_SCORE_ZIP, 4) AS AVG_SCORE_ZIP,
    round(t.AVG_SCORE_BORO_CD, 4) AS AVG_SCORE_BORO_CD,
    round(
        CASE
            WHEN t.AVG_SCORE_ZIP IS NULL OR t.AVG_SCORE_ZIP = 0 THEN NULL
            ELSE t.NUMBER_PER_ZIP / t.AVG_SCORE_ZIP
        END,
        4
    ) AS RESTAURANT_DENSITY_QUALITY_INDEX,
    t.LANDUSE,
    CASE
        WHEN t.YEARBUILT > 0 AND t.YEARBUILT <= {CURRENT_YEAR}
        THEN {CURRENT_YEAR} - t.YEARBUILT
        ELSE NULL
    END AS BUILDING_AGE,
    t.YEARBUILT,
    round(cs.AVG_CRIME_PER_ZIP, 4) AS AVG_CRIME_PER_ZIP,
    CAST(ca.COUNT_CRIME_PER_ZIP AS BIGINT) AS COUNT_CRIME_PER_ZIP,
    ca.DOMINANT_CRIME_TYPE,
    round(
        {zscore_sql('ca.COUNT_CRIME_PER_ZIP', 'cs.AVG_CRIME_PER_ZIP', 'cs.STDEV_CRIME_PER_ZIP')}
        +
        {zscore_sql('t.AVG_SCORE_ZIP', 'ss.MEAN_AVG_SCORE_ZIP', 'ss.STDEV_AVG_SCORE_ZIP')},
        4
    ) AS CRIME_INSPECTION_RISK_SCORE
FROM temp1 t
LEFT JOIN crime_aligned ca ON t.ZIPCODE = ca.ZIPCODE
CROSS JOIN stdev_per_zip sz
CROSS JOIN stdev_per_boro sb
CROSS JOIN crime_stats cs
CROSS JOIN score_stats ss
"""


temp2_sql = build_temp2_sql()

## 8. Etap SQL TEMP3

Etap `TEMP3` tworzy finalną tabelę wynikową z kolumnami używanymi w raporcie projektu.

In [ ]:
def build_temp3_sql():
    norm_score_zip = zscore_sql("t.SCORE", "t.AVG_SCORE_ZIP", "sz.STDEV_SCORE_ZIP")
    norm_crime_zip = zscore_sql("t.COUNT_CRIME_PER_ZIP", "t.AVG_CRIME_PER_ZIP", "sc.STDEV_CRIME_PER_ZIP")
    cuisine_relative = zscore_sql("t.SCORE", "t.AVG_SCORE_BORO_CD", "sbc.STDEV_SCORE_BORO_CD")

    return f"""
CREATE OR REPLACE TABLE {TEMP3_TABLE} AS
WITH temp2 AS (
    SELECT
        CAMIS,
        upper(trim(coalesce(CAST(BORO AS STRING), ''))) AS BORO,
        {first_five_digits_sql('ZIPCODE')} AS ZIPCODE,
        ADDRESS,
        upper(trim(coalesce(CAST(CUISINE_DESCRIPTION AS STRING), ''))) AS CUISINE_DESCRIPTION,
        {safe_double_sql('SCORE')} AS SCORE,
        {safe_double_sql('AVG_SCORE_ZIP')} AS AVG_SCORE_ZIP,
        {safe_double_sql('AVG_SCORE_BORO_CD')} AS AVG_SCORE_BORO_CD,
        {safe_double_sql('AVG_CRIME_PER_ZIP')} AS AVG_CRIME_PER_ZIP,
        {safe_double_sql('COUNT_CRIME_PER_ZIP')} AS COUNT_CRIME_PER_ZIP,
        YEARBUILT,
        LANDUSE,
        BUILDING_AGE,
        RESTAURANT_DENSITY_QUALITY_INDEX,
        DOMINANT_CRIME_TYPE
    FROM {TEMP2_TABLE}
),
stdev_score_zip AS (
    SELECT ZIPCODE, stddev_pop(SCORE) AS STDEV_SCORE_ZIP
    FROM temp2
    GROUP BY ZIPCODE
),
stdev_score_boro_cd AS (
    SELECT BORO, CUISINE_DESCRIPTION, stddev_pop(SCORE) AS STDEV_SCORE_BORO_CD
    FROM temp2
    GROUP BY BORO, CUISINE_DESCRIPTION
),
stdev_crime AS (
    SELECT coalesce(stddev_pop(COUNT_CRIME_PER_ZIP), 0.0) AS STDEV_CRIME_PER_ZIP
    FROM (
        SELECT DISTINCT ZIPCODE, COUNT_CRIME_PER_ZIP
        FROM temp2
    )
)
SELECT
    t.CAMIS,
    t.BORO,
    t.ZIPCODE,
    t.ADDRESS,
    t.CUISINE_DESCRIPTION,
    t.YEARBUILT,
    t.LANDUSE,
    round(({norm_score_zip}) + ({norm_crime_zip}), 4) AS CRIME_INSPECTION_RISK_SCORE,
    t.BUILDING_AGE AS BUILDING_AGE_SCORE,
    t.RESTAURANT_DENSITY_QUALITY_INDEX,
    round({cuisine_relative}, 4) AS CUISINE_RELATIVE_SCORE,
    t.DOMINANT_CRIME_TYPE
FROM temp2 t
LEFT JOIN stdev_score_zip sz
    ON t.ZIPCODE = sz.ZIPCODE
LEFT JOIN stdev_score_boro_cd sbc
    ON t.BORO = sbc.BORO
   AND t.CUISINE_DESCRIPTION = sbc.CUISINE_DESCRIPTION
CROSS JOIN stdev_crime sc
"""


temp3_sql = build_temp3_sql()

## 9. Uruchomienie całego pipeline SQL

Ustaw `RUN_SQL_PIPELINE = True`, aby wykonać wszystkie etapy.

In [ ]:
RUN_SQL_PIPELINE = True

if RUN_SQL_PIPELINE:
    pipeline_start = perf_counter()
    stage_results = []

    elapsed, rows, status = run_timed_sql("TEMP1_SQL", temp1_sql, TEMP1_TABLE)
    stage_results.append(("TEMP1_SQL", elapsed, rows, status))

    elapsed, rows, status = run_timed_sql("NYPD_ZIP_SQL", nypd_zip_sql, NYPD_ZIP_TABLE)
    stage_results.append(("NYPD_ZIP_SQL", elapsed, rows, status))

    # TEMP2 używa tabel powstałych we wcześniejszych etapach, dlatego budujemy SQL ponownie po ich utworzeniu.
    temp2_sql = build_temp2_sql()
    elapsed, rows, status = run_timed_sql("TEMP2_SQL", temp2_sql, TEMP2_TABLE)
    stage_results.append(("TEMP2_SQL", elapsed, rows, status))

    temp3_sql = build_temp3_sql()
    elapsed, rows, status = run_timed_sql("TEMP3_SQL", temp3_sql, TEMP3_TABLE)
    stage_results.append(("TEMP3_SQL", elapsed, rows, status))

    print("\n===== PODSUMOWANIE CZASÓW PIPELINE SQL =====")
    total = 0.0
    for stage_name, elapsed, rows, status in stage_results:
        total += elapsed
        print(f"{stage_name}: {format_duration(elapsed)} ({elapsed:.3f} s), rows={rows}, status={status}")
    print(f"Łączny czas etapów SQL: {format_duration(total)} ({total:.3f} s)")

    pipeline_elapsed = perf_counter() - pipeline_start
    print(f"Całkowity czas uruchomienia pipeline SQL: {format_duration(pipeline_elapsed)} ({pipeline_elapsed:.3f} s)")
else:
    print("RUN_SQL_PIPELINE = False. Ustaw True, aby uruchomić wszystkie etapy.")

## 10. Weryfikacja wyniku w SQL

Ta komórka pokazuje końcową tabelę `TEMP3` i prostą agregację kontrolną.

In [ ]:
print("Tabela końcowa:", TEMP3_TABLE)

temp3_count = spark.sql(f"SELECT COUNT(*) AS rows FROM {TEMP3_TABLE}").first()["rows"]
print("Liczba rekordów TEMP3 SQL:", temp3_count)

display(spark.sql(f"""
SELECT *
FROM {TEMP3_TABLE}
LIMIT 20
"""))

display(spark.sql(f"""
SELECT
    BORO,
    COUNT(*) AS RECORDS,
    round(avg(CRIME_INSPECTION_RISK_SCORE), 4) AS AVG_RISK_SCORE,
    round(avg(CUISINE_RELATIVE_SCORE), 4) AS AVG_CUISINE_RELATIVE_SCORE,
    round(avg(RESTAURANT_DENSITY_QUALITY_INDEX), 4) AS AVG_RESTAURANT_DENSITY_QUALITY_INDEX
FROM {TEMP3_TABLE}
GROUP BY BORO
ORDER BY BORO
"""))

## 11. Zapytania do wykonania w edytorze SQL Databricks

Po uruchomieniu notebooka można wejść w **SQL Editor** i wykonać:

```sql
SELECT COUNT(*) FROM workspace.default.test_temp3_latest_sql;

SELECT *
FROM workspace.default.test_temp3_latest_sql
LIMIT 20;

SELECT
    BORO,
    COUNT(*) AS RECORDS,
    ROUND(AVG(CRIME_INSPECTION_RISK_SCORE), 4) AS AVG_RISK_SCORE
FROM workspace.default.test_temp3_latest_sql
GROUP BY BORO
ORDER BY BORO;
```

Dla pełnych danych usuńcie prefiks `test_`, czyli użyjcie tabeli:

```sql
workspace.default.temp3_latest_sql
```

## 12. Opcjonalny eksport wyników SQL z powrotem do CSV

Wyniki podstawowo są tabelami Delta w bazie Databricks. Jeżeli potrzebny jest eksport do plików CSV w volume, można użyć poniższej funkcji.

In [ ]:
def export_table_to_csv_dir(table_name: str, output_dir: str):
    print(f"Eksport tabeli {table_name} do {output_dir}")
    (
        spark.table(table_name)
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(output_dir)
    )
    print("STATUS: OK")


EXPORT_RESULTS_TO_CSV = False

if EXPORT_RESULTS_TO_CSV:
    export_table_to_csv_dir(TEMP1_TABLE, f"{RESULTS_DIR}/temp1_latest_sql_csv")
    export_table_to_csv_dir(NYPD_ZIP_TABLE, f"{RESULTS_DIR}/nypd_zip_latest_sql_csv")
    export_table_to_csv_dir(TEMP2_TABLE, f"{RESULTS_DIR}/temp2_latest_sql_csv")
    export_table_to_csv_dir(TEMP3_TABLE, f"{RESULTS_DIR}/temp3_latest_sql_csv")
else:
    print("EXPORT_RESULTS_TO_CSV = False. Wyniki pozostają jako tabele Delta w bazie Databricks.")